In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/apache1_dataset.csv")

print(df.shape)

(104557, 67)


In [2]:
df["CUSTOMERSTATUS"].value_counts(dropna=False)

CUSTOMERSTATUS
ALIVE           91673
DEAD            11900
NaN               958
BROUGHT DEAD       23
LIVE                2
0                   1
Name: count, dtype: int64

In [3]:
df["CUSTOMERSTATUS"] = df["CUSTOMERSTATUS"].replace({
    "ALIVE": 0,
    "LIVE": 0,
    "DEAD": 1,
    "BROUGHT DEAD": 1
})

df = df[df["CUSTOMERSTATUS"].isin([0, 1])]

print(df["CUSTOMERSTATUS"].value_counts())
print("\nShape:", df.shape)

CUSTOMERSTATUS
0    91675
1    11923
Name: count, dtype: int64

Shape: (103598, 67)


In [4]:
drop_cols = [
    "ID",
    "UHID",
    "IPNumber",
    "ICUChartDate",
    "CreatedBy",
    "CreatedDate",
    "UpdatedBy",
    "UpdatedDate",
    "DOB",
    "AdmittingDoctor",
    "Ward",
    "RNK",
    "CREATEDON",
    "LOCATIONID",
    "DISCHARGEDATE",
    "PERIOD_WID",

    # Leakage Columns
    "ApacheivScore",
    "ApsScore",
    "EstimatedMortalityRate",
    "EstimatedLengthOfStay"
]

df = df.drop(columns=drop_cols)

print("Remaining Columns:", len(df.columns))
print(df.columns.tolist())

Remaining Columns: 47
['Age', 'Temperature', 'MeanArterialPressure', 'HeartRate', 'RespiratoryRate', 'FiO2', 'pO2', 'pCO2', 'ArterialpH', 'Sodium', 'UrineOutput', 'Creatinine', 'Urea', 'BSL', 'Albumin', 'Bilirubin', 'Hematocrit', 'WBC', 'IsGCSNotAvailable', 'GCSEyes', 'GCSVerbal', 'GCSMotor', 'MecanicalVentilation', 'CRF', 'Lymphoma', 'Cirrhosis', 'Leukemia', 'HepaticFailure', 'Immunosuppression', 'MetastaticCarcinoma', 'AIDS', 'PreICULengthOfStay', 'DiagnosisType', 'Origin', 'EmergencySurgery', 'Readmission', 'Thrombolysis', 'RespiratoryQuotient', 'AtmosphericPressure', 'SystemValue', 'DiagnosisValue', 'Gender', 'AdmissionDate', 'UNIT_ID', 'APACHE_WARD', 'CUSTOMERSTATUS', 'LOCATION']


In [4]:
cat_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:")
print(cat_cols.tolist())

Categorical Columns:
['UHID', 'IPNumber', 'ICUChartDate', 'MecanicalVentilation', 'SystemValue', 'DiagnosisValue', 'CreatedDate', 'DOB', 'Gender', 'AdmittingDoctor', 'AdmissionDate', 'CREATEDON', 'APACHE_WARD', 'CUSTOMERSTATUS', 'LOCATION', 'DISCHARGEDATE']


In [6]:
# Convert admission date

df["AdmissionDate"] = pd.to_datetime(
    df["AdmissionDate"],
    errors="coerce"
)

df["AdmissionYear"] = df["AdmissionDate"].dt.year
df["AdmissionMonth"] = df["AdmissionDate"].dt.month

df.drop("AdmissionDate", axis=1, inplace=True)

print(df[["AdmissionYear","AdmissionMonth"]].head())

   AdmissionYear  AdmissionMonth
0           2026               1
1           2026               2
2           2026               2
3           2026               1
4           2026               1


/var/folders/zv/n5ygljn517v155nmdp1knl0h0000gn/T/ipykernel_74120/3669379344.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["AdmissionDate"] = pd.to_datetime(


In [5]:
df.isnull().sum().sort_values(ascending=False).head(20)

UpdatedBy              103598
Ward                   103598
UpdatedDate            103598
DISCHARGEDATE           11019
APACHE_WARD              4506
LOCATION                    7
LOCATIONID                  7
AtmosphericPressure         0
CreatedDate                 0
CreatedBy                   0
DiagnosisValue              0
SystemValue                 0
ID                          0
RespiratoryQuotient         0
Thrombolysis                0
EmergencySurgery            0
Origin                      0
DiagnosisType               0
Readmission                 0
AdmittingDoctor             0
dtype: int64

In [6]:
cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:
    print("\n")
    print(col)
    print("Unique Values:", df[col].nunique())
    



UHID
Unique Values: 93961


IPNumber
Unique Values: 103598


ICUChartDate
Unique Values: 1240


MecanicalVentilation
Unique Values: 2


SystemValue
Unique Values: 12


DiagnosisValue
Unique Values: 101


CreatedDate
Unique Values: 97932


DOB
Unique Values: 23821


Gender
Unique Values: 3


AdmittingDoctor
Unique Values: 2870


AdmissionDate
Unique Values: 1265


CREATEDON
Unique Values: 8


APACHE_WARD
Unique Values: 743


CUSTOMERSTATUS
Unique Values: 2


LOCATION
Unique Values: 43


DISCHARGEDATE
Unique Values: 26484


In [7]:
df = df.drop(columns=["APACHE_WARD"])

print(df.shape)

(103598, 66)


In [8]:
X = df.drop("CUSTOMERSTATUS", axis=1)
y = df["CUSTOMERSTATUS"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (103598, 65)
Target: (103598,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (82878, 65)
Test : (20720, 65)


In [10]:
print(y_train.value_counts(normalize=True))

CUSTOMERSTATUS
0    0.884915
1    0.115085
Name: proportion, dtype: float64


In [11]:
!pip install xgboost -q


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [12]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1
Note: you may need to restart the kernel to use updated packages.


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cat_cols = [
    "MecanicalVentilation",
    "SystemValue",
    "DiagnosisValue",
    "Gender",
    "LOCATION"
]

num_cols = [c for c in X.columns if c not in cat_cols]

print("Numerical:", len(num_cols))
print("Categorical:", len(cat_cols))

Numerical: 60
Categorical: 5


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            cat_cols
        )
    ],
    remainder="passthrough"
)

In [15]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

In [17]:
# Automatically detect all categorical columns

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Categorical Columns:")
print(cat_cols)

print("\nCount:", len(cat_cols))

Categorical Columns:
['UHID', 'IPNumber', 'ICUChartDate', 'MecanicalVentilation', 'SystemValue', 'DiagnosisValue', 'CreatedDate', 'DOB', 'Gender', 'AdmittingDoctor', 'AdmissionDate', 'CREATEDON', 'LOCATION', 'DISCHARGEDATE']

Count: 14


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            cat_cols
        )
    ],
    remainder="passthrough"
)

In [19]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    enable_categorical=False
)

In [20]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Pipeline Ready")

Pipeline Ready


In [21]:
pipeline.fit(X_train, y_train)

print("Training Complete")

Training Complete


In [24]:
# Fix target datatype

y_train = y_train.astype(int)
y_test = y_test.astype(int)

print(y_train.dtype)
print(y_test.dtype)

int64
int64


In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("="*50)
print("MORTALITY MODEL RESULTS")
print("="*50)

print("Accuracy :", round(accuracy_score(y_test, preds), 4))
print("Precision:", round(precision_score(y_test, preds), 4))
print("Recall   :", round(recall_score(y_test, preds), 4))
print("F1 Score :", round(f1_score(y_test, preds), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, probs), 4))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

MORTALITY MODEL RESULTS
Accuracy : 0.891
Precision: 0.6171
Recall   : 0.1392
F1 Score : 0.2272
ROC AUC  : 0.8164

Confusion Matrix
[[18129   206]
 [ 2053   332]]

Classification Report
              precision    recall  f1-score   support

           0       0.90      0.99      0.94     18335
           1       0.62      0.14      0.23      2385

    accuracy                           0.89     20720
   macro avg       0.76      0.56      0.58     20720
weighted avg       0.87      0.89      0.86     20720



In [26]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print("Negative:", neg)
print("Positive:", pos)
print("Scale Pos Weight:", scale_pos_weight)

Negative: 73340
Positive: 9538
Scale Pos Weight: 7.6892430278884465


In [27]:
from xgboost import XGBClassifier

balanced_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=7.689,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

print("Balanced Model Created")

Balanced Model Created


In [28]:
from sklearn.pipeline import Pipeline

balanced_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", balanced_model)
])

print("Balanced Pipeline Ready")

Balanced Pipeline Ready


In [29]:
balanced_pipeline.fit(X_train, y_train)

print("Balanced Training Complete")

Balanced Training Complete


In [30]:
balanced_preds = balanced_pipeline.predict(X_test)
balanced_probs = balanced_pipeline.predict_proba(X_test)[:, 1]

In [31]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("="*60)
print("BALANCED MORTALITY MODEL")
print("="*60)

print("Accuracy :", round(accuracy_score(y_test, balanced_preds), 4))
print("Precision:", round(precision_score(y_test, balanced_preds), 4))
print("Recall   :", round(recall_score(y_test, balanced_preds), 4))
print("F1 Score :", round(f1_score(y_test, balanced_preds), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, balanced_probs), 4))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, balanced_preds))

print("\nClassification Report")
print(classification_report(y_test, balanced_preds))

BALANCED MORTALITY MODEL
Accuracy : 0.7795
Precision: 0.2944
Recall   : 0.6553
F1 Score : 0.4063
ROC AUC  : 0.8172

Confusion Matrix
[[14589  3746]
 [  822  1563]]

Classification Report
              precision    recall  f1-score   support

           0       0.95      0.80      0.86     18335
           1       0.29      0.66      0.41      2385

    accuracy                           0.78     20720
   macro avg       0.62      0.73      0.64     20720
weighted avg       0.87      0.78      0.81     20720



In [32]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(
    y_test,
    balanced_probs
)

f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

best_idx = np.argmax(f1_scores)

print("Best Threshold :", thresholds[best_idx])
print("Best Precision :", precision[best_idx])
print("Best Recall    :", recall[best_idx])
print("Best F1 Score  :", f1_scores[best_idx])

Best Threshold : 0.5920252
Best Precision : 0.3522510231923602
Best Recall    : 0.5412997903563941
Best F1 Score  : 0.42677685945637034


In [33]:
feature_names = balanced_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

import pandas as pd

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": balanced_pipeline.named_steps[
        "model"
    ].feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance.head(30))

                                                  Feature  Importance
290284                  remainder__EstimatedMortalityRate    0.003773
290282                           remainder__ApacheivScore    0.001963
160573                        cat__MecanicalVentilation_b    0.001079
160572                        cat__MecanicalVentilation_a    0.001018
290268                     remainder__MetastaticCarcinoma    0.000964
266482           cat__LOCATION_Apollo Loga Hospital Karur    0.000937
266307                    cat__AdmissionDate_7/31/23 0:00    0.000912
266494          cat__LOCATION_Bilaspur - Lingyadi Village    0.000908
265801                   cat__AdmissionDate_11/13/24 0:00    0.000875
160547                     cat__ICUChartDate_9/24/25 0:00    0.000843
160157                     cat__ICUChartDate_2/24/25 0:00    0.000815
266507         cat__LOCATION_Reach Hospital - Karim Nagar    0.000802
264968  cat__AdmittingDoctor_UNIT1 NEPHROLOGY(DR.T.DAS...    0.000798
262517  cat__Admitti

In [34]:
print(X_train.columns.tolist())

['ID', 'UHID', 'IPNumber', 'ICUChartDate', 'Age', 'Temperature', 'MeanArterialPressure', 'HeartRate', 'RespiratoryRate', 'FiO2', 'pO2', 'pCO2', 'ArterialpH', 'Sodium', 'UrineOutput', 'Creatinine', 'Urea', 'BSL', 'Albumin', 'Bilirubin', 'Hematocrit', 'WBC', 'IsGCSNotAvailable', 'GCSEyes', 'GCSVerbal', 'GCSMotor', 'MecanicalVentilation', 'CRF', 'Lymphoma', 'Cirrhosis', 'Leukemia', 'HepaticFailure', 'Immunosuppression', 'MetastaticCarcinoma', 'AIDS', 'PreICULengthOfStay', 'DiagnosisType', 'Origin', 'EmergencySurgery', 'Readmission', 'Thrombolysis', 'RespiratoryQuotient', 'AtmosphericPressure', 'SystemValue', 'DiagnosisValue', 'CreatedBy', 'CreatedDate', 'UpdatedBy', 'UpdatedDate', 'DOB', 'Gender', 'AdmittingDoctor', 'Ward', 'ApacheivScore', 'ApsScore', 'EstimatedMortalityRate', 'EstimatedLengthOfStay', 'AdmissionDate', 'UNIT_ID', 'CREATEDON', 'RNK', 'LOCATIONID', 'LOCATION', 'DISCHARGEDATE', 'PERIOD_WID']


In [35]:
drop_cols = [
    "ID",
    "UHID",
    "IPNumber",
    "ICUChartDate",
    "CreatedBy",
    "CreatedDate",
    "UpdatedBy",
    "UpdatedDate",
    "DOB",
    "AdmittingDoctor",
    "Ward",
    "RNK",
    "CREATEDON",
    "LOCATIONID",
    "DISCHARGEDATE",
    "PERIOD_WID",

    # APACHE-IV Leakage
    "ApacheivScore",
    "ApsScore",
    "EstimatedMortalityRate",
    "EstimatedLengthOfStay"
]

df_clean = df.drop(columns=drop_cols)

print(df_clean.shape)
print(df_clean.columns.tolist())

(103598, 46)
['Age', 'Temperature', 'MeanArterialPressure', 'HeartRate', 'RespiratoryRate', 'FiO2', 'pO2', 'pCO2', 'ArterialpH', 'Sodium', 'UrineOutput', 'Creatinine', 'Urea', 'BSL', 'Albumin', 'Bilirubin', 'Hematocrit', 'WBC', 'IsGCSNotAvailable', 'GCSEyes', 'GCSVerbal', 'GCSMotor', 'MecanicalVentilation', 'CRF', 'Lymphoma', 'Cirrhosis', 'Leukemia', 'HepaticFailure', 'Immunosuppression', 'MetastaticCarcinoma', 'AIDS', 'PreICULengthOfStay', 'DiagnosisType', 'Origin', 'EmergencySurgery', 'Readmission', 'Thrombolysis', 'RespiratoryQuotient', 'AtmosphericPressure', 'SystemValue', 'DiagnosisValue', 'Gender', 'AdmissionDate', 'UNIT_ID', 'CUSTOMERSTATUS', 'LOCATION']


In [36]:
# Remove non-clinical columns

df_clean = df_clean.drop(
    columns=[
        "AdmissionDate",
        "UNIT_ID",
        "LOCATION"
    ]
)

print(df_clean.shape)

(103598, 43)


In [37]:
X = df_clean.drop("CUSTOMERSTATUS", axis=1)
y = df_clean["CUSTOMERSTATUS"].astype(int)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("\nCategorical Columns:")
print(cat_cols)

X Shape: (103598, 42)
y Shape: (103598,)

Categorical Columns:
['MecanicalVentilation', 'SystemValue', 'DiagnosisValue', 'Gender']


In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(82878, 42)
(20720, 42)


In [39]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print("Negative:", neg)
print("Positive:", pos)
print("Scale Pos Weight:", scale_pos_weight)

Negative: 73340
Positive: 9538
Scale Pos Weight: 7.6892430278884465


In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            [
                "MecanicalVentilation",
                "SystemValue",
                "DiagnosisValue",
                "Gender"
            ]
        )
    ],
    remainder="passthrough"
)

In [41]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

In [42]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

print("Training Complete")

Training Complete


In [43]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

preds = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)[:, 1]

print("="*60)
print("CLEAN APACHE-I MORTALITY MODEL")
print("="*60)

print("Accuracy :", round(accuracy_score(y_test, preds), 4))
print("Precision:", round(precision_score(y_test, preds), 4))
print("Recall   :", round(recall_score(y_test, preds), 4))
print("F1 Score :", round(f1_score(y_test, preds), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, probs), 4))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

CLEAN APACHE-I MORTALITY MODEL
Accuracy : 0.7982
Precision: 0.306
Recall   : 0.5941
F1 Score : 0.404
ROC AUC  : 0.8064

Confusion Matrix
[[15122  3213]
 [  968  1417]]

Classification Report
              precision    recall  f1-score   support

           0       0.94      0.82      0.88     18335
           1       0.31      0.59      0.40      2385

    accuracy                           0.80     20720
   macro avg       0.62      0.71      0.64     20720
weighted avg       0.87      0.80      0.82     20720



In [44]:
import joblib

joblib.dump(
    pipeline,
    "../models/apache1_mortality_model.pkl"
)

print("Mortality Model Saved")

Mortality Model Saved
